# SoccerTrack v2 — treinamento BAS

Treina o classificador temporal do baseline oficial a partir de features MViT-V1-B pré-treinadas em Kinetics-400. O extrator é **congelado**; somente o spotter temporal é treinado. O spotter é inspirado em T-DEED, não a implementação original. Execute as células em ordem no Colab com GPU ou Jupyter local com CUDA.

Obtenha acesso ao [dataset SoccerTrack v2](https://huggingface.co/datasets/atomscott/soccertrack-v2), aceite as condições e baixe vídeos e anotações da release **v1.2** diretamente para o disco local do ambiente. A extração em vídeos panorâmicos completos é demorada e demanda bastante espaço. No Colab, os dados baixados em `/content` são temporários; o cache de features, o checkpoint e as predições são salvos no Google Drive. Os pesos MViT são baixados pelo torchvision na primeira execução.

A anotação v1.2 usa `position` em milissegundos a partir do primeiro frame do vídeo de cada tempo. Verifique visualmente o alinhamento antes de treinar. O baseline usa um único rótulo por amostra: eventos simultâneos podem perder um dos rótulos.


In [1]:
from pathlib import Path
import os, subprocess, sys, json, random

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = Path("/content" if IS_COLAB else Path.cwd()).resolve()
# Dados no disco temporário do Colab; resultados no Google Drive.
DATA_ROOT = Path("/content/soccertrack-v2" if IS_COLAB else "./soccertrack-v2-data").resolve()
OUTPUT_ROOT = Path("/content/drive/MyDrive/soccertrack-bas-run" if IS_COLAB else "./soccertrack-bas-run").resolve()
REPO = WORKSPACE / "SoccerTrack-v2-baseline"
REVISION = "a752d294df2a0e816e827b70f284980698f5a50e"
print("Dados:", DATA_ROOT, "\nResultados:", OUTPUT_ROOT)


Mounted at /content/drive
Dados: /content/soccertrack-v2 
Resultados: /content/drive/MyDrive/soccertrack-bas-run


## Código e dependências

O clone é fixado em um commit; o checkout é recusado se houver alterações locais. Instale PyTorch e torchvision compatíveis com seu CUDA; no Colab os pacotes normalmente já estão instalados. Reinicie o runtime se alterar versões já importadas.


In [2]:
if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(f"{REPO} já existe e não é um clone Git")
    subprocess.run(["git", "clone", "https://github.com/AtomScott/SoccerTrack-v2.git", str(REPO)], check=True)
if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip():
    raise RuntimeError("O clone contém alterações locais; use outro diretório.")
subprocess.run(["git", "fetch", "origin", REVISION], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
# Instale somente as dependências necessárias ao BAS. O runtime deve ter torch e torchvision compatíveis.
subprocess.run([sys.executable, "-m", "pip", "install", "numpy", "opencv-python-headless", "pyyaml", "tqdm", "huggingface_hub[cli]"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'numpy', 'opencv-python-headless', 'pyyaml', 'tqdm', 'huggingface_hub[cli]'], returncode=0)

In [3]:
import cv2
import numpy as np
import torch
import torchvision
from tqdm.auto import tqdm
from dataclasses import asdict

if not torch.cuda.is_available():
    raise RuntimeError("CUDA não disponível. Selecione GPU no Colab ou configure PyTorch com CUDA.")
# Evita importar o pacote local src do ProjetoPDS em vez do pacote upstream.
if "src" in sys.modules and not str(getattr(sys.modules["src"], "__file__", "")).startswith(str(REPO)):
    raise RuntimeError("Outro pacote 'src' já foi importado; reinicie o kernel e execute este notebook primeiro.")
sys.path.insert(0, str(REPO))
from baselines.bas.train import (FeatureCfg, SpotterCfg, build_mvit_extractor, build_spotter,
    labels_to_clip_targets, make_windows, train_spotter, predict_clip_probs, decode_predictions, write_predictions)
from src.data_utils.soccertrack_v2 import BAS_LABELS, Event
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__, "Torchvision:", torchvision.__version__)


GPU: NVIDIA L4 Torch: 2.14.1+cu130 Torchvision: 0.29.1+cu130


## Divisão das partidas

Edite as listas antes do download. A divisão padrão usa todas as dez partidas e pode exceder o espaço disponível no Colab; para experimentar, escolha menos partidas, mantendo treino, validação e teste não vazios e disjuntos.


In [4]:
TRAIN = ["117092", "118575", "118576"]
VAL = ["117093", "132877"]
TEST = ["128057", "132831"]
SEED = 42
feature_cfg = FeatureCfg()
spot_cfg = SpotterCfg()
# Ajuste conforme a memória disponível. As features usam a mesma transformação do MViT oficial.
FEATURE_BATCH = 4
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
all_ids = TRAIN + VAL + TEST
if not all((TRAIN, VAL, TEST)) or len(all_ids) != len(set(all_ids)):
    raise ValueError("As divisões devem ser não vazias, sem partidas repetidas.")



## Download direto para o ambiente

Aceite primeiro as condições na [página do dataset](https://huggingface.co/datasets/atomscott/soccertrack-v2). Execute a autenticação usando um token de leitura autorizado; não inclua o token no notebook. O download seleciona apenas as anotações BAS e os vídeos das partidas configuradas, na revisão `v1.2`. Para usar arquivos existentes, defina `DOWNLOAD_DATA = False` e confirme que são da mesma revisão.

O dry run mostra arquivos e tamanhos antes da transferência. Compare o tamanho total com o espaço disponível; o dataset completo é grande e o Colab pode não ter espaço suficiente. Selecione menos partidas ou use armazenamento com capacidade suficiente. Execute a célula seguinte para iniciar o download.


In [5]:
DOWNLOAD_DATA = True
DATASET_REVISION = "v1.2"
if DOWNLOAD_DATA:
    import shutil
    from huggingface_hub import login
    # Abre um campo de token no notebook; o token não deve ser salvo nas células.
    login(add_to_git_credential=False)
    hf = shutil.which("hf")
    if hf is None:
        raise RuntimeError("CLI hf não encontrada; execute novamente a célula de dependências.")
    patterns = [pattern for mid in all_ids for pattern in (f"bas/{mid}/*", f"videos/{mid}/*")]
    download_command = [hf, "download", "atomscott/soccertrack-v2", "--type", "dataset",
                        "--revision", DATASET_REVISION, "--local-dir", str(DATA_ROOT),
                        "--include", *patterns]
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    free = shutil.disk_usage(DATA_ROOT).free / 1024**3
    print(f"Espaço livre no disco: {free:.1f} GiB")
    subprocess.run(download_command + ["--dry-run"], check=True)


Espaço livre no disco: 235.4 GiB


In [6]:
if DOWNLOAD_DATA:
    # Verifique o tamanho acima antes de executar esta célula.
    subprocess.run(download_command, check=True)
    print("Download concluído em", DATA_ROOT)
else:
    print("Usando dados existentes em", DATA_ROOT)


Download concluído em /content/soccertrack-v2


## Verificação dos dados

São exigidos um JSON BAS e dois vídeos por partida. Os nomes panorâmicos da distribuição são reconhecidos; ajuste `video_path` se a cópia local usar outros nomes. Confirme que as anotações v1.2 e os vídeos estão alinhados.


In [12]:
def video_path(mid, half):
    # Atualize esta lista caso a cópia local use outros nomes.
    names = ([f"{mid}_panorama_1st_half.mp4", f"{mid}_1st.mp4", f"{mid}_1.mp4", "1.mp4", "1st_half.mp4"] if half == 1 else
             [f"{mid}_panorama_2nd_half.mp4", f"{mid}_2nd.mp4", f"{mid}_2.mp4", "2.mp4", "2nd_half.mp4"])
    matches = [DATA_ROOT / "videos" / mid / name for name in names if (DATA_ROOT / "videos" / mid / name).is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Esperado exatamente um vídeo para {mid}, tempo {half}; encontrados: {matches}. Ajuste video_path se necessário.")
    return matches[0]

def annotation_path(mid):
    return DATA_ROOT / "bas" / mid / f"{mid}_12_class_events.json"

TRAIN = ["132877", "118575", "118576"]
VAL = ["117093", "132877"]
TEST = ["128057", "132831"]

all_ids = TRAIN + VAL + TEST

for mid in all_ids:
    if not annotation_path(mid).is_file():
        raise FileNotFoundError(annotation_path(mid))
    for half in (1, 2):
        print(mid, half, video_path(mid, half))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


132877 1 /content/soccertrack-v2/videos/132877/132877_panorama_1st_half.mp4
132877 2 /content/soccertrack-v2/videos/132877/132877_panorama_2nd_half.mp4
118575 1 /content/soccertrack-v2/videos/118575/118575_panorama_1st_half.mp4
118575 2 /content/soccertrack-v2/videos/118575/118575_panorama_2nd_half.mp4
118576 1 /content/soccertrack-v2/videos/118576/118576_panorama_1st_half.mp4
118576 2 /content/soccertrack-v2/videos/118576/118576_panorama_2nd_half.mp4
117093 1 /content/soccertrack-v2/videos/117093/117093_panorama_1st_half.mp4
117093 2 /content/soccertrack-v2/videos/117093/117093_panorama_2nd_half.mp4
132877 1 /content/soccertrack-v2/videos/132877/132877_panorama_1st_half.mp4
132877 2 /content/soccertrack-v2/videos/132877/132877_panorama_2nd_half.mp4
128057 1 /content/soccertrack-v2/videos/128057/128057_panorama_1st_half.mp4
128057 2 /content/soccertrack-v2/videos/128057/128057_panorama_2nd_half.mp4
132831 1 /content/soccertrack-v2/videos/132831/132831_panorama_1st_half.mp4
132831 2 /co

## Extração de features

Os frames são lidos sequencialmente e apenas um lote de clipes é mantido em memória. Cada cache registra configuração, commit, tamanho e data do vídeo; alterações nessas entradas invalidam o cache. Se substituir um vídeo preservando tamanho e data, apague manualmente o cache.


In [13]:
extractor, transform = build_mvit_extractor(feature_cfg, pretrained=True)
extractor = extractor.to("cuda").eval()

def features_for(mid, half):
    path = video_path(mid, half)
    stat = path.stat()
    meta = {"revision": REVISION, "video": str(path), "size": stat.st_size,
            "mtime_ns": stat.st_mtime_ns, "features": asdict(feature_cfg)}
    cache = OUTPUT_ROOT / "features" / f"{mid}_{half}.npz"
    cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        with np.load(cache, allow_pickle=False) as data:
            if json.loads(str(data["meta"].item())) == meta:
                return data["features"], data["centers"]
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir {path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or count <= 0:
        cap.release(); raise ValueError(f"FPS ou número de frames inválido: {path}")
    nclips = max(0, int(np.floor((count / fps - feature_cfg.clip_s) / feature_cfg.stride_s)) + 1)
    if not nclips:
        cap.release(); raise ValueError(f"Vídeo curto demais: {path}")
    # Apenas os frames necessários são decodificados; os índices podem ser compartilhados por clipes.
    span = int(round(feature_cfg.clip_s * fps))
    starts = np.rint(np.arange(nclips) * feature_cfg.stride_s * fps).astype(int)
    indices = [np.linspace(s, s + span - 1, feature_cfg.num_frames).round().astype(int) for s in starts]
    centers = np.rint((np.arange(nclips) * feature_cfg.stride_s + feature_cfg.clip_s / 2) * 1000).astype(np.int64)
    # Map each requested frame to all clips using it; overlapping clips are supported.
    requested = {}
    for clip_id, frame_ids in enumerate(indices):
        for frame_id in set(map(int, frame_ids)):
            requested.setdefault(frame_id, []).append(clip_id)
    result, batch, active = [], [], {}
    next_clip = 0
    try:
        with torch.no_grad():
            for frame_idx in tqdm(range(int(indices[-1][-1]) + 1), desc=f"{mid} tempo {half}"):
                ok, frame = cap.read()
                if not ok:
                    raise RuntimeError(f"Falha lendo frame {frame_idx} de {path}")
                if frame_idx not in requested:
                    continue
                rgb = torch.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)
                for clip_id in requested[frame_idx]:
                    active.setdefault(clip_id, {})[frame_idx] = rgb
                while next_clip < nclips and frame_idx == indices[next_clip][-1]:
                    frames = active.pop(next_clip)
                    clip = torch.stack([frames[int(i)] for i in indices[next_clip]])
                    batch.append(transform(clip))
                    next_clip += 1
                    if len(batch) == FEATURE_BATCH or next_clip == nclips:
                        result.append(extractor(torch.stack(batch).to("cuda")).cpu().numpy())
                        batch.clear()
    finally:
        cap.release()
    if next_clip != nclips:
        raise RuntimeError(f"Leitura incompleta: {next_clip}/{nclips} clipes")
    features = np.concatenate(result).astype(np.float32)
    np.savez_compressed(cache, features=features, centers=centers, meta=json.dumps(meta, sort_keys=True))
    return features, centers


Downloading: "https://download.pytorch.org/models/mvit_v1_b-dbeb1030.pth" to /root/.cache/torch/hub/checkpoints/mvit_v1_b-dbeb1030.pth


100%|██████████| 140M/140M [00:00<00:00, 236MB/s]


## Treinamento

O parser abaixo usa diretamente `position` por tempo da release v1.2, sem aplicar o deslocamento de 45 minutos presente em propriedades antigas do loader upstream. Confirme que os dados realmente pertencem à release v1.2. O baseline atribui ao centro de cada clipe o evento mais próximo dentro de 500 ms e usa cross-entropy com pesos de classe.


In [16]:
def events_for(mid):
    path = annotation_path(mid)
    raw = json.loads(path.read_text())

    if not isinstance(raw, dict):
        raise ValueError(f"Expected a JSON object in {path}")

    if "match_id" in raw and str(raw["match_id"]) != str(mid):
        raise ValueError(f"Wrong match ID in {path}: {raw['match_id']}")

    items = raw.get("actions", raw.get("annotations"))
    if not isinstance(items, list):
        raise ValueError(f"No actions/annotations list in {path}: {list(raw)}")

    label_map = {label.casefold(): label for label in BAS_LABELS}
    events = []

    for item in items:
        try:
            half = int(item["gameTime"].split(" - ")[0])
            t_ms = int(item["position"])
            label = label_map[item["label"].strip().casefold()]
        except (KeyError, ValueError, TypeError, AttributeError) as exc:
            raise ValueError(f"Invalid event in {path}: {item}") from exc

        if half not in (1, 2) or t_ms < 0:
            raise ValueError(f"Invalid event time in {path}: {item}")

        events.append(Event(half=half, clock="", t_ms=t_ms, label=label))

    return events

windows, targets = [], []
for mid in TRAIN:
    events = events_for(mid)
    for half in (1, 2):
        feats, centers = features_for(mid, half)
        y = labels_to_clip_targets(events, centers, half)
        xw, yw = make_windows(feats, y, spot_cfg.window)
        windows.append(xw); targets.append(yw)
        print(mid, half, "clipes:", len(feats), "positivos:", int((y > 0).sum()))
spotter = build_spotter(feature_cfg, spot_cfg)
spotter = train_spotter(spotter, np.concatenate(windows), np.concatenate(targets), spot_cfg, device="cuda", verbose=True)
checkpoint = {"state_dict": spotter.cpu().state_dict(), "feature_cfg": asdict(feature_cfg),
              "spotter_cfg": asdict(spot_cfg), "labels": list(BAS_LABELS), "revision": REVISION,
              "dataset_release": "v1.2", "train": TRAIN, "val": VAL, "test": TEST, "seed": SEED}
torch.save(checkpoint, OUTPUT_ROOT / "bas_spotter.pt")
print("Checkpoint:", OUTPUT_ROOT / "bas_spotter.pt")


132877 tempo 1:   0%|          | 0/67700 [00:00<?, ?it/s]

132877 1 clipes: 2707 positivos: 1128


132877 tempo 2:   0%|          | 0/69350 [00:00<?, ?it/s]

132877 2 clipes: 2773 positivos: 1027


118575 tempo 1:   0%|          | 0/71850 [00:00<?, ?it/s]

118575 1 clipes: 2873 positivos: 1016


... (2 outputs left out)

118576 tempo 1:   0%|          | 0/72825 [00:00<?, ?it/s]

118576 1 clipes: 2912 positivos: 1062


118576 tempo 2:   0%|          | 0/72775 [00:00<?, ?it/s]

118576 2 clipes: 2910 positivos: 948
[bas] epoch 1/20 loss=2.4320
[bas] epoch 2/20 loss=2.2191
[bas] epoch 3/20 loss=1.9162
[bas] epoch 4/20 loss=1.3958
[bas] epoch 5/20 loss=0.9519
[bas] epoch 6/20 loss=0.7260
[bas] epoch 7/20 loss=0.5568
[bas] epoch 8/20 loss=0.4223
[bas] epoch 9/20 loss=0.4417
[bas] epoch 10/20 loss=0.4062
[bas] epoch 11/20 loss=0.3308
[bas] epoch 12/20 loss=0.3654
[bas] epoch 13/20 loss=0.2342
[bas] epoch 14/20 loss=0.2425
[bas] epoch 15/20 loss=0.4596
[bas] epoch 16/20 loss=0.1979
[bas] epoch 17/20 loss=0.1484
[bas] epoch 18/20 loss=0.1527
[bas] epoch 19/20 loss=0.2683
[bas] epoch 20/20 loss=0.1421
Checkpoint: /content/drive/MyDrive/soccertrack-bas-run/bas_spotter.pt


In [15]:
import json

path = annotation_path(TRAIN[0])
data = json.loads(path.read_text())

print("File:", path)
print("Top-level type:", type(data).__name__)

if isinstance(data, dict):
    print("Keys:", list(data.keys()))
    for key, value in data.items():
        print(f"\n{key}: {type(value).__name__}")
        if isinstance(value, list) and value:
            print("First item:", value[0])
        elif isinstance(value, dict):
            print("Nested keys:", list(value.keys())[:20])

File: /content/soccertrack-v2/bas/132877/132877_12_class_events.json
Top-level type: dict
Keys: ['match_id', 'fps', 'clock', 'actions']

match_id: str

fps: float

clock: str

actions: list
First item: {'gameTime': '1 - 0:00', 'label': 'DRIVE', 'position': '720', 'frame': 19, 'team': 'left', 'player_id': '518295'}


## Predições de validação e teste

O checkpoint contém o spotter; na próxima sessão, recrie também o extrator MViT com os pesos Kinetics indicados acima. Use validação para ajustar limiar e NMS antes de olhar o teste. O avaliador upstream dessa revisão ordena predições por tempo em vez de confiança, então seu mAP **não é confiável**. Exporte as predições e use um avaliador BAS validado e ordenado por confiança antes de publicar métricas.


In [19]:
# Em uma nova sessão, execute as células de configuração e extração e carregue o checkpoint.
if "spotter" not in globals():
    checkpoint = torch.load(OUTPUT_ROOT / "bas_spotter.pt", map_location="cpu", weights_only=True)
    if checkpoint["revision"] != REVISION or checkpoint["labels"] != list(BAS_LABELS):
        raise ValueError("Checkpoint incompatível com este baseline.")
    feature_cfg = FeatureCfg(**checkpoint["feature_cfg"])
    spot_cfg = SpotterCfg(**checkpoint["spotter_cfg"])
    spotter = build_spotter(feature_cfg, spot_cfg)
    spotter.load_state_dict(checkpoint["state_dict"])

# Para avaliar apenas validação, use VAL; depois de escolher os parâmetros, use TEST.
PREDICT_MATCHES = VAL
for mid in PREDICT_MATCHES:
    predicted = []
    for half in (1, 2):
        feats, centers = features_for(mid, half)
        probs = predict_clip_probs(spotter, feats, device="cuda")
        predicted.extend(decode_predictions(probs, centers, half,
            nms_window_s=spot_cfg.nms_window_s, min_confidence=spot_cfg.min_confidence,
            require_above_background=spot_cfg.require_above_background,
            max_peaks_per_class=spot_cfg.max_peaks_per_class))
    path = write_predictions(predicted, OUTPUT_ROOT / "predictions", mid)
    print(mid, len(predicted), path)


117093 tempo 1:   0%|          | 0/67625 [00:00<?, ?it/s]

117093 tempo 2:   0%|          | 0/70475 [00:00<?, ?it/s]

117093 2583 /content/drive/MyDrive/soccertrack-bas-run/predictions/117093/117093_12_class_events.json
132877 2548 /content/drive/MyDrive/soccertrack-bas-run/predictions/132877/132877_12_class_events.json
